In [1]:
import torchvision
import torch
from PIL import Image
from sklearn.metrics import confusion_matrix, accuracy_score
import torch.nn as nn
from torch import optim
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import time
import random

In [2]:
# Replace last classifier to only handle 2 cases, one benign one high grade
num_classes = 2
model = torchvision.models.alexnet(weights = 'AlexNet_Weights.DEFAULT')
# replace the last classifier
model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

In [3]:
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier[6].parameters():
    param.requires_grad = True

In [4]:
labels = pd.read_csv('case_grade_match.csv')

In [5]:
# Need to group the patches by their cases, and also need to randomly split, the function below will group the cases
def group_patches(patch_dir):
    case_patches = {}
    for filename in os.listdir(patch_dir):
        # If the image size is less than a kilobyte, don't use the patch
        if os.path.getsize(os.path.join(patch_dir, filename)) < 2000:
            continue
        # If 'patched' in filename, not an actual patch
        if 'patched_' in filename:
            continue
        if 'h&e' not in filename:
            continue
        elif filename.endswith('.png'):
            case_num = int(filename.split('_')[1])
            if case_num not in case_patches:
                case_patches[case_num] = []
            case_patches[case_num].append(os.path.join(patch_dir, filename))
    return case_patches

class PNGDataset(Dataset):
    def __init__(self, case_patches, labels_df, transform=None):
        self.case_patches = case_patches
        self.labels_df = labels_df
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for case_num, patches in case_patches.items():
            label = labels_df.loc[labels_df['Case'] == case_num, 'Class'].values[0]
            label = 0 if label == 1 else 1
            for patch_path in patches:
                self.image_paths.append(patch_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        # Get patches, if they are not patch, don't use the patch
        image = Image.open(image_path).convert('RGB')
        # Get the label information using the labels dataframe based on case number
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        return image, label

# Group the patches
patches = group_patches('Patches/')

# Get case numbers and their labels
case_nums = list(patches.keys())
dataset = labels.loc[[(int(x)-1) for x in case_nums]]
# Remove those that are equal to 2
noindex = dataset.Class != 2.0
X = dataset[noindex].Case
y = dataset[noindex].Class
train, test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify = y, random_state=40)

# Create the training patches and the test patches
train_patches = {case_num: patches[int(case_num)] for case_num in train}
test_patches = {case_num: patches[int(case_num)] for case_num in test}


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Generate the actual datasets for the models
train_dataset = PNGDataset(train_patches, labels, transform=transform)
test_dataset = PNGDataset(test_patches, labels, transform = transform)
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)


In [6]:
start = time.time()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 1000  # Number of epochs
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

patience = 0
best_loss = 100000

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)


        optimizer.zero_grad()


        outputs = model(images)
        loss = criterion(outputs, labels)


        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_dataloader)
    if epoch_loss < best_loss:
        #print('Model Saved!')
        #torch.save(model.state_dict(), f'model_epoch_{epoch + 1}.pth')
        best_loss = epoch_loss
        patience = 0
    else:
        patience += 1
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}, Patience: {patience}, Best Loss: {best_loss:.4f}')
    
    if patience >= 50:
        print(f'Early stopping at epoch {epoch + 1}')
        break
end = time.time()
elapsed = end - start
minutes = int(elapsed // 60)
seconds = int(elapsed % 60)
print(f'Time taken: {minutes} minutes, {seconds} seconds')

Epoch [1/1000], Loss: 0.6037, Patience: 0, Best Loss: 0.6037
Epoch [2/1000], Loss: 0.4669, Patience: 0, Best Loss: 0.4669
Epoch [3/1000], Loss: 0.4286, Patience: 0, Best Loss: 0.4286
Epoch [4/1000], Loss: 0.4137, Patience: 0, Best Loss: 0.4137
Epoch [5/1000], Loss: 0.3900, Patience: 0, Best Loss: 0.3900
Epoch [6/1000], Loss: 0.3754, Patience: 0, Best Loss: 0.3754
Epoch [7/1000], Loss: 0.3623, Patience: 0, Best Loss: 0.3623
Epoch [8/1000], Loss: 0.3618, Patience: 0, Best Loss: 0.3618
Epoch [9/1000], Loss: 0.3498, Patience: 0, Best Loss: 0.3498
Epoch [10/1000], Loss: 0.3446, Patience: 0, Best Loss: 0.3446
Epoch [11/1000], Loss: 0.3494, Patience: 1, Best Loss: 0.3446
Epoch [12/1000], Loss: 0.3466, Patience: 2, Best Loss: 0.3446
Epoch [13/1000], Loss: 0.3390, Patience: 0, Best Loss: 0.3390
Epoch [14/1000], Loss: 0.3486, Patience: 1, Best Loss: 0.3390
Epoch [15/1000], Loss: 0.3283, Patience: 0, Best Loss: 0.3283
Epoch [16/1000], Loss: 0.3183, Patience: 0, Best Loss: 0.3183
Epoch [17/1000], 

In [7]:
pred = []
labels = []
with torch.no_grad():
    for images, label in test_dataloader:
        images, label = images.to(device), label.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        labels.append(label)
        pred.append(predicted)

pred = torch.cat(pred).cpu()
labels = torch.cat(labels).cpu()
accuracy = accuracy_score(labels, pred)
print(f'Accuracy: {accuracy}')
confusion_matrix(labels, pred)

Accuracy: 0.6127770534550195


array([[122, 267],
       [327, 818]], dtype=int64)